## ColdCode：面向初学者的智能代码辅助编程专家

此版本已将 Prompt、长代码分析、模型调用、结果提取、文件流等核心逻辑迁移到 `coldcode_core`，notebook 仅保留 JupyterLab 前端壳。


In [ ]:
import ipywidgets as W
from IPython.display import display, HTML as DHTML
import httpx, json, re, time, hashlib, datetime, os, shutil
from pathlib import Path
import time

In [ ]:
# ===== ColdCode 模块化核心导入 =====

import sys
from pathlib import Path

_SEARCH_ROOTS = [
    Path.cwd(),
    Path.cwd() / "coldcode_modular",
    Path.cwd().parent,
]
for _root in _SEARCH_ROOTS:
    if (_root / "src").exists():
        _root_str = str(_root)
        if _root_str not in sys.path:
            sys.path.insert(0, _root_str)

from src import (
    MODEL_FAST,
    MODEL_STRONG,
    MODE_HELP,
    LAST_OUTPUT,
    normalize_file_path,
    load_text_file,
    apply_fixed_code_to_file,
    restore_backup_file,
    build_prompt_compare_text,
    export_markdown_result,
    export_tech_report,
    run_task_stream,
)


In [8]:
# Prompt、长代码分析、模型调用、提取器等核心逻辑已迁移到 coldcode_core。
# 该 notebook 现在只保留 JupyterLab 前端壳与按钮事件。


In [9]:
# ===== UI 设计部分 =====
# ===================== 全局 CSS 美化 =====================
display(DHTML("""
<style>
/* 整体观感 */
.cc-card {
    background: #ffffff;
    border: 1px solid #e2e8f0;
    border-radius: 18px;
    padding: 16px 18px;
    box-shadow: 0 6px 18px rgba(15, 23, 42, 0.06);
    margin-bottom: 14px;
}

.cc-header-card {
    background: linear-gradient(135deg, #f8fbff 0%, #eef6ff 100%);
    border: 1px solid #dbeafe;
    border-radius: 20px;
    padding: 20px 24px;
    box-shadow: 0 8px 20px rgba(59, 130, 246, 0.08);
    margin-bottom: 14px;
}

.cc-section-title {
    font-size: 15px;
    font-weight: 700;
    color: #334155;
    margin: 2px 0 12px 0;
    letter-spacing: 0.2px;
}

.cc-field-label {
    font-size: 14px;
    font-weight: 700;
    color: #334155;
    margin: 4px 0 6px 2px;
}

.cc-subtext {
    font-size: 13px;
    color: #64748b;
    margin-top: 2px;
}

.cc-status-chip {
    display: inline-block;
    padding: 6px 12px;
    border-radius: 999px;
    font-size: 13px;
    font-weight: 700;
    background: #eff6ff;
    color: #1d4ed8;
    border: 1px solid #bfdbfe;
}

.cc-tip-box {
    background: #f8fafc;
    border: 1px solid #e2e8f0;
    border-radius: 14px;
    padding: 10px 12px;
}

.cc-kbd {
    display: inline-block;
    padding: 1px 6px;
    border-radius: 8px;
    border: 1px solid #cbd5e1;
    background: #f8fafc;
    font-family: monospace;
    font-size: 12px;
}

/* widget 通用润色 */
.jupyter-widgets .widget-button {
    border-radius: 12px !important;
    font-weight: 700 !important;
    box-shadow: none !important;
}

.jupyter-widgets .widget-text input,
.jupyter-widgets .widget-textarea textarea,
.jupyter-widgets .widget-dropdown select {
    border-radius: 12px !important;
    border: 1px solid #cbd5e1 !important;
    box-shadow: none !important;
    padding-left: 8px !important;
}

.jupyter-widgets .widget-text input:focus,
.jupyter-widgets .widget-textarea textarea:focus,
.jupyter-widgets .widget-dropdown select:focus {
    border-color: #60a5fa !important;
    box-shadow: 0 0 0 3px rgba(96, 165, 250, 0.15) !important;
}

/* Output 区域 */
.cc-output-box {
    background: #0f172a;
    color: #e2e8f0 !important;
    border: 1px solid #1e293b;
    border-radius: 16px;
    padding: 10px 12px;
}

.cc-output-box * {
    color: #e2e8f0 !important;
}

.cc-output-box pre,
.cc-output-box code,
.cc-output-box .output,
.cc-output-box .output_text,
.cc-output-box .output_subarea,
.cc-output-box .jp-OutputArea-output,
.cc-output-box .jp-RenderedText,
.cc-output-box .jp-OutputArea-child {
    color: #e2e8f0 !important;
    background: transparent !important;
}

/* 让 VBox/HBox 卡片更自然 */
.cc-no-bottom {
    margin-bottom: 0 !important;
}
</style>
"""))


# ===================== 小工具函数 =====================
def section_title(text):
    return W.HTML(f"<div class='cc-section-title'>{text}</div>")

def field_label(text):
    return W.HTML(f"<div class='cc-field-label'>{text}</div>")

# ===================== 你的原始控件（保留变量名） =====================

# 标题
title = W.HTML(
    value="""
    <div style="text-align:center;">
        <div style="
            font-size:38px;
            font-weight:800;
            color:#1e293b;
            line-height:1.2;
            margin-bottom:6px;
            letter-spacing:0.3px;
        ">
            💦 Coldrain's ColdCode
        </div>
        <div style="
            font-size:15px;
            color:#64748b;
            margin-bottom:12px;
        ">
            Debug · Explain · Refactor · Scaffold/Test · ROCm Doctor
        </div>
        <div style="
            height:3px;
            border-radius:999px;
            background:linear-gradient(to right, #38bdf8, #60a5fa, #818cf8);
            width:100%;
        "></div>
    </div>
    """
)

# mode 选择框
mode_dd = W.Dropdown(
    options=["Debug", "Explain", "Refactor", "Scaffold/Test", "ROCm Doctor"],
    value="Debug",
    description="🚀模式:",
    layout=W.Layout(width='240px', margin='4px 10px 4px 0')
)

# 语言选择框
lang_dd = W.Dropdown(
    options=["Python", "Java", "C++"],
    value="Python",
    description="📖语言:",
    layout=W.Layout(width='220px', margin='4px 10px 4px 0')
)

# 模型选择框
model_dd = W.Dropdown(
    options=[("快速 llama3.1:8b", MODEL_FAST), ("高质量 qwen3-coder:30b", MODEL_STRONG)],
    value=MODEL_FAST,
    description="🤖模型:",
    layout=W.Layout(width='260px', margin='4px 10px 4px 0')
)

# prompt version 选择框
prompt_ver_dd = W.Dropdown(
    options=[("v0 朴素", "v0"), ("v1 结构化", "v1"), ("v2 few-shot", "v2"), ("v3 工程化", "v3")],
    value="v3",
    description="🚄Prompt:",
    layout=W.Layout(width='220px', margin='4px 10px 4px 0')
)

# 文件工作流控件
file_path_in = W.Text(
    placeholder="例如：src/main.py 或 ./demo.py",
    description="📁文件:",
    layout=W.Layout(width="100%", height='40px')
)

load_file_btn = W.Button(
    description="Load File",
    button_style="",
    icon="folder-open",
    layout=W.Layout(width='130px', height='40px', margin='4px 8px 4px 0'),
    tooltip="读取文件内容到代码框"
)

apply_file_btn = W.Button(
    description="Apply to File",
    button_style="info",
    icon="save",
    layout=W.Layout(width='140px', height='40px', margin='4px 8px 4px 0'),
    tooltip="把修复后代码写回文件"
)

restore_file_btn = W.Button(
    description="Restore Backup",
    button_style="",
    icon="history",
    layout=W.Layout(width='150px', height='40px', margin='4px 8px 4px 0'),
    tooltip="用 .bak 备份恢复文件"
)

file_info = W.HTML(
    value="<div class='cc-subtext'>当前未加载文件。你也可以只把代码粘贴到下方代码框中。</div>",
    layout=W.Layout(width='100%', margin='2px 0 8px 0')
)

# 参数微调框
num_predict_sl = W.IntSlider(
    value=350, min=120, max=2000, step=10,
    description="输出上限:",
    continuous_update=False,
    layout=W.Layout(width='360px', margin='4px 16px 4px 0')
)

temp_sl = W.FloatSlider(
    value=0.2, min=0.0, max=0.8, step=0.05,
    description="temperature:",
    continuous_update=False,
    layout=W.Layout(width='380px', margin='4px 0 4px 0')
)

# 输入框
question_in = W.Text(
    placeholder=MODE_HELP["Debug"]["question_placeholder"],
    description="",
    layout=W.Layout(width="100%", height='40px')
)

code_in = W.Textarea(
    placeholder=MODE_HELP["Debug"]["code_placeholder"],
    description="",
    layout=W.Layout(width="100%", height="240px")
)

tb_in = W.Textarea(
    placeholder=MODE_HELP["Debug"]["tb_placeholder"],
    description="",
    layout=W.Layout(width="100%", height="160px")
)

mode_tip = W.HTML(
    value="",
    layout=W.Layout(width='100%', margin='0 0 6px 0')
)

learning_card_ck = W.Checkbox(
    value=True,
    description="Debug 后追加错误成长卡（仅 Debug 生效）",
    indent=False,
    layout=W.Layout(width='350px', margin='4px 16px 4px 0')
)

# 按钮
run_btn = W.Button(
    description="Run",
    button_style="warning",
    icon="play",
    layout=W.Layout(width='120px', height='40px', margin='4px 8px 4px 0'),
    tooltip="运行当前请求"
)

apply_btn = W.Button(
    description="Apply Fixed Code",
    button_style="info",
    icon="check",
    layout=W.Layout(width='160px', height='40px', margin='4px 8px 4px 0'),
    tooltip="应用修复后的代码"
)

undo_btn = W.Button(
    description="Undo",
    button_style="",
    icon="rotate-left",
    layout=W.Layout(width='110px', height='40px', margin='4px 8px 4px 0'),
    tooltip="撤销上一次应用"
)

clear_btn = W.Button(
    description="Clear",
    button_style="",
    icon="trash",
    layout=W.Layout(width='110px', height='40px', margin='4px 8px 4px 0'),
    tooltip="清空输入输出"
)

export_btn = W.Button(
    description="Export .md",
    button_style="success",
    icon="download",
    layout=W.Layout(width='130px', height='40px', margin='4px 8px 4px 0'),
    tooltip="导出 Markdown"
)

report_btn = W.Button(
    description="Export Report",
    button_style="primary",
    icon="file-text",
    layout=W.Layout(width='140px', height='40px', margin='4px 8px 4px 0'),
    tooltip="导出报告"
)

compare_btn = W.Button(
    description="Prompt Compare",
    button_style="",
    icon="columns",
    layout=W.Layout(width='160px', height='40px', margin='4px 8px 4px 0'),
    tooltip="对比当前模式在不同 Prompt 版本下的差异"
)

demo_btn = W.Button(
    description="Fill Demo",
    button_style="",
    icon="magic",
    layout=W.Layout(width='130px', height='40px', margin='4px 0 4px 0'),
    tooltip="自动填充当前模式的演示输入"
)

apply_btn.disabled = True
apply_file_btn.disabled = True
restore_file_btn.disabled = True
undo_btn.disabled = True

status = W.HTML(
    value="<span class='cc-status-chip'>Ready</span>",
    layout=W.Layout(width="100%", margin='0 0 10px 0')
)

out = W.Output(
    layout=W.Layout(
        width='100%',
        height='460px',
        overflow='auto',
        border='none'
    )
)

prompt_lab_out = W.Output(
    layout=W.Layout(
        width='100%',
        height='220px',
        overflow='auto',
        border='none'
    )
)

# ===================== 控件 description 宽度统一 =====================
for w in [lang_dd, mode_dd, model_dd, prompt_ver_dd, file_path_in]:
    w.style.description_width = '72px'

for w in [num_predict_sl, temp_sl]:
    w.style.description_width = '88px'

# ===================== 顶部页眉 =====================
header_box = W.VBox([title], layout=W.Layout(width='100%'))
header_box.add_class("cc-header-card")

# ===================== 控制区 =====================
selectbar_row = W.HBox(
    [lang_dd, mode_dd, model_dd, prompt_ver_dd],
    layout=W.Layout(
        width='100%',
        flex_flow='row wrap',
        align_items='center',
        justify_content='flex-start'
    )
)

btn_row = W.HBox(
    [run_btn, apply_btn, apply_file_btn, undo_btn, restore_file_btn, clear_btn, export_btn, report_btn],
    layout=W.Layout(
        width='100%',
        flex_flow='row wrap',
        align_items='center',
        justify_content='flex-start'
    )
)

advanced_row = W.HBox(
    [learning_card_ck, compare_btn, demo_btn],
    layout=W.Layout(
        width='100%',
        flex_flow='row wrap',
        align_items='center',
        justify_content='flex-start'
    )
)

param_row = W.HBox(
    [num_predict_sl, temp_sl],
    layout=W.Layout(
        width='100%',
        flex_flow='row wrap',
        align_items='center',
        justify_content='flex-start'
    )
)

control_panel = W.VBox(
    [
        section_title("⚙️ 控制台"),
        selectbar_row,
        section_title("🔧 功能"),
        btn_row,
        W.HTML("<div style='height:8px;'></div>"),
        section_title("✨ 亮点展示"),
        advanced_row,
        W.HTML("<div style='height:8px;'></div>"),
        section_title("🎛️ 参数微调"),
        param_row,
    ],
    layout=W.Layout(width='100%')
)
control_panel.add_class("cc-card")

# ===================== 左侧输入区 =====================
left_panel = W.VBox(
    [
        section_title("📝 输入区"),
        mode_tip,
        field_label("📁 工作文件"),
        file_path_in,
        file_info,
        W.HBox([load_file_btn], layout=W.Layout(width='100%', justify_content='flex-start')),
        W.HTML("<div style='height:8px;'></div>"),
        field_label("❓ 问题说明"),
        question_in,
        W.HTML("<div style='height:8px;'></div>"),
        field_label("💻 代码内容 / 安装命令"),
        code_in,
        W.HTML("<div style='height:8px;'></div>"),
        field_label("⚠️ 报错 / 日志 / 环境信息"),
        tb_in,
    ],
    layout=W.Layout(
        flex='1 1 720px',
        width='50%',
        min_width='680px'
    )
)
left_panel.add_class("cc-card")

# ===================== 右侧输出区 =====================
output_title = W.HTML("""
<div class='cc-section-title'>
    📦 输出区
    <div class='cc-subtext'>模型响应、修复建议、补丁结果都会显示在这里</div>
</div>
""")

output_wrap = W.VBox([out], layout=W.Layout(width='100%'))
output_wrap.add_class("cc-output-box")

prompt_lab_title = W.HTML("""
<div class='cc-section-title'>
    🧪 Prompt Lab
    <div class='cc-subtext'>点击 Prompt Compare，对比当前模式在 v0-v3 下的提示词差异</div>
</div>
""")

prompt_lab_wrap = W.VBox([prompt_lab_out], layout=W.Layout(width='100%'))
prompt_lab_wrap.add_class("cc-output-box")

right_panel = W.VBox(
    [
        output_title,
        status,
        output_wrap,
        W.HTML("<div style='height:12px;'></div>"),
        prompt_lab_title,
        prompt_lab_wrap
    ],
    layout=W.Layout(
        flex='1 1 720px',
        width='50%',
        min_width='680px'
    )
)
right_panel.add_class("cc-card")

# ===================== 主工作区 =====================
workspace = W.HBox(
    [left_panel, right_panel],
    layout=W.Layout(
        width='100%',
        flex_flow='row wrap',
        align_items='stretch',
        justify_content='space-between'
    )
)


In [10]:
def on_clear(_):
    """
    清空输出区，不清空输入内容。
    """
    out.clear_output()
    status.value = "<span class='cc-status-chip'>Ready</span>"
    with prompt_lab_out:
        prompt_lab_out.clear_output()
        print("点击 Prompt Compare 查看当前模式在 v0-v3 下的提示词差异。")


def update_file_info(path: str = "", *, message: str = "", ok: bool = True):
    if path:
        safe = path.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
        color = "#0f766e" if ok else "#b45309"
        extra = f"<div class='cc-subtext' style='margin-top:4px;color:{color};'>{message}</div>" if message else ""
        file_info.value = f"<div class='cc-subtext'><b>当前文件：</b><code>{safe}</code></div>{extra}"
    else:
        file_info.value = "<div class='cc-subtext'>当前未加载文件。你也可以只把代码粘贴到下方代码框中。</div>"


def render_mode_tip(mode: str):
    help_pack = MODE_HELP[mode]
    badge = "🩺 ROCm 问诊" if mode == "ROCm Doctor" else f"✨ {mode} 模式"
    mode_tip.value = (
        "<div class='cc-tip-box'>"
        f"<div style='font-weight:700;color:#0f172a;margin-bottom:4px;'>{badge}</div>"
        f"<div class='cc-subtext' style='margin-top:0;color:#475569;'>{help_pack['hint']}</div>"
        "</div>"
    )


def sync_placeholders(mode: str):
    help_pack = MODE_HELP[mode]
    question_in.placeholder = help_pack["question_placeholder"]
    code_in.placeholder = help_pack["code_placeholder"]
    tb_in.placeholder = help_pack["tb_placeholder"]


def on_mode_change(change=None):
    mode = mode_dd.value
    render_mode_tip(mode)
    sync_placeholders(mode)
    learning_card_ck.disabled = (mode != "Debug")
    if mode != "Debug":
        learning_card_ck.tooltip = "该开关仅在 Debug 模式下生效"
    else:
        learning_card_ck.tooltip = "开启后会在调试结果末尾追加错误成长卡"


def on_compare_prompt(_):
    text = build_prompt_compare_text(mode_dd.value)
    with prompt_lab_out:
        prompt_lab_out.clear_output()
        print(text)


def on_fill_demo(_):
    mode = mode_dd.value
    if mode == "Debug":
        lang_dd.value = "Python"
        question_in.value = "请帮我定位问题，给最小修复步骤、diff 和修复后代码。"
        code_in.value = "def add(a, b):\n    return a + b\n\nprint(ad(1, 2))\n"
        tb_in.value = (
            'Traceback (most recent call last):\n'
            '  File "demo.py", line 4, in <module>\n'
            '    print(ad(1, 2))\n'
            "NameError: name 'ad' is not defined\n"
        )
        status.value = "<b style='color:green'>已填充 Debug 演示样例。</b>"
        return

    if mode == "Explain":
        lang_dd.value = "Python"
        question_in.value = "请按总览、逐段解释、关键概念来讲清楚这段代码。"
        code_in.value = (
            "def moving_average(nums, window=3):\n"
            "    if window <= 0:\n"
            "        raise ValueError('window must be positive')\n"
            "    result = []\n"
            "    for i in range(len(nums)):\n"
            "        left = max(0, i - window + 1)\n"
            "        chunk = nums[left:i+1]\n"
            "        result.append(sum(chunk) / len(chunk))\n"
            "    return result\n"
        )
        tb_in.value = ""
        status.value = "<b style='color:green'>已填充 Explain 演示样例。</b>"
        return

    if mode == "Refactor":
        lang_dd.value = "Python"
        question_in.value = "请做最小重构，提高可读性，不要改变功能。"
        code_in.value = (
            "def score(data):\n"
            "    s = 0\n"
            "    for i in range(len(data)):\n"
            "        if data[i] > 60:\n"
            "            s = s + data[i]\n"
            "    return s\n"
        )
        tb_in.value = ""
        status.value = "<b style='color:green'>已填充 Refactor 演示样例。</b>"
        return

    if mode == "Scaffold/Test":
        lang_dd.value = "Python"
        question_in.value = "请生成一个命令行 Todo 小项目，带 pytest 测试和运行说明。"
        code_in.value = ""
        tb_in.value = "要求：支持 add/list/done 三个命令；目录结构清晰；尽量最小可运行。"
        status.value = "<b style='color:green'>已填充 Scaffold/Test 演示样例。</b>"
        return

    if mode == "ROCm Doctor":
        lang_dd.value = "Python"
        question_in.value = "我在 Ubuntu 上装了 ROCm 和 PyTorch，rocminfo 能看到 GPU，但 torch.cuda.is_available() 返回 False，帮我判断最可能的问题并给最小排查步骤。"
        code_in.value = (
            "python -c \"import torch; "
            "print(torch.__version__); "
            "print(torch.version.hip); "
            "print(torch.cuda.is_available())\""
        )
        tb_in.value = (
            "$ rocminfo | head -n 5\n"
            "ROCk module is loaded\n"
            "=====================\n"
            "HSA Agents:\n"
            "  Agent 1: gfx1100\n\n"
            "$ python -c \"import torch; print(torch.__version__); print(torch.version.hip); print(torch.cuda.is_available())\"\n"
            "2.3.0\n"
            "None\n"
            "False\n\n"
            "$ python -m pip show torch\n"
            "Name: torch\n"
            "Version: 2.3.0\n"
            "Summary: Tensors and Dynamic neural networks in Python\n"
        )
        status.value = "<b style='color:green'>已填充 ROCm Doctor 演示样例。</b>"
        return


def on_load_file(_):
    try:
        info = load_text_file(file_path_in.value)
        code_in.value = info["content"]
        file_path_in.value = info["path"]
        LAST_OUTPUT["loaded_file_path"] = info["path"]
        LAST_OUTPUT["backup_file_path"] = info.get("backup_path", "")
        apply_file_btn.disabled = True
        restore_file_btn.disabled = not info.get("backup_exists", False)
        update_file_info(info["path"], message="文件已加载到代码框。", ok=True)
        status.value = f"<b style='color:green'>已加载文件：</b> <code>{info['path']}</code>"
    except Exception as e:
        update_file_info("", message="加载失败。", ok=False)
        apply_file_btn.disabled = True
        restore_file_btn.disabled = True
        status.value = f"<b style='color:#b00'>加载失败：</b> {e}"


def on_apply_file(_):
    fixed = LAST_OUTPUT.get("fixed_code", "").strip()
    if not fixed:
        status.value = "<b style='color:#b00'>没有检测到修复后代码，请先 Run 并确保输出包含修复后代码。</b>"
        return

    path = normalize_file_path(file_path_in.value or LAST_OUTPUT.get("loaded_file_path", ""))
    if not path:
        status.value = "<b style='color:#b00'>请先输入并加载目标文件，再执行 Apply to File。</b>"
        return

    try:
        info = apply_fixed_code_to_file(path, fixed)
        code_in.value = info["content"]
        file_path_in.value = info["path"]
        LAST_OUTPUT["loaded_file_path"] = info["path"]
        LAST_OUTPUT["backup_file_path"] = info["backup_path"]
        LAST_OUTPUT["last_apply_target"] = "file"
        restore_file_btn.disabled = False
        apply_file_btn.disabled = False
        update_file_info(info["path"], message=f"已写回文件，并创建备份：{Path(info['backup_path']).name}", ok=True)
        status.value = f"<b style='color:green'>已写回文件 ✅</b> <code>{info['path']}</code>（备份：<code>{Path(info['backup_path']).name}</code>）"
    except Exception as e:
        status.value = f"<b style='color:#b00'>写回失败：</b> {e}"


def on_restore_backup(_):
    path = normalize_file_path(file_path_in.value or LAST_OUTPUT.get("loaded_file_path", ""))
    if not path:
        status.value = "<b style='color:#b00'>请先指定文件路径。</b>"
        return

    try:
        info = restore_backup_file(path)
        code_in.value = info["content"]
        file_path_in.value = info["path"]
        LAST_OUTPUT["loaded_file_path"] = info["path"]
        LAST_OUTPUT["backup_file_path"] = info["backup_path"]
        LAST_OUTPUT["last_apply_target"] = "restore_backup"
        restore_file_btn.disabled = False
        update_file_info(info["path"], message=f"已从备份恢复：{Path(info['backup_path']).name}", ok=True)
        status.value = f"<b style='color:green'>已恢复备份 ✅</b> <code>{info['path']}</code>"
    except Exception as e:
        status.value = f"<b style='color:#b00'>恢复失败：</b> {e}"


def on_export(_):
    try:
        result_file, log_file = export_markdown_result('.')
        status.value = f"<b style='color:green'>已导出：</b> <code>{result_file.name}</code> 和 <code>{log_file.name}</code>（当前目录）"
    except Exception as e:
        status.value = f"<b style='color:#b00'>导出失败：</b> {e}"


def on_export_report(_):
    try:
        report_file = export_tech_report('.')
        status.value = f"<b style='color:green'>已导出技术文档骨架：</b> <code>{report_file.name}</code>"
    except Exception as e:
        status.value = f"<b style='color:#b00'>导出失败：</b> {e}"


def on_apply(_):
    fixed = LAST_OUTPUT.get("fixed_code", "").strip()
    if not fixed:
        status.value = "<b style='color:#b00'>没有检测到修复后代码（请确保输出里有“## 修复后代码/重构后代码”下的代码块）。</b>"
        return

    current = code_in.value or ""
    if not current.strip():
        status.value = "<b style='color:#b00'>代码框为空，仍然可以应用，但建议先放入原始代码便于对比。</b>"

    LAST_OUTPUT["prev_code"] = current
    code_in.value = fixed + ("\n" if not fixed.endswith("\n") else "")
    undo_btn.disabled = False
    apply_file_btn.disabled = (normalize_file_path(file_path_in.value or LAST_OUTPUT.get("loaded_file_path", "")) == "")
    LAST_OUTPUT["last_apply_target"] = "code_box"
    status.value = "<b style='color:green'>已用“修复后代码”覆盖代码框 ✅</b>（如需写回真实文件，请点击 Apply to File）"


def on_undo(_):
    prev = LAST_OUTPUT.get("prev_code", "")
    if not prev:
        status.value = "<b style='color:#b00'>没有可撤销的历史。</b>"
        return
    code_in.value = prev
    LAST_OUTPUT["prev_code"] = ""
    undo_btn.disabled = True
    status.value = "<b style='color:green'>已撤销代码框修改 ✅</b>"


def on_run(_):
    run_btn.disabled = True
    apply_btn.disabled = True
    out.clear_output()
    status.value = "<span class='cc-status-chip'>Running...</span>"

    active_file_path = normalize_file_path(file_path_in.value or LAST_OUTPUT.get("loaded_file_path", ""))

    try:
        with out:
            result = run_task_stream(
                lang=lang_dd.value,
                mode=mode_dd.value,
                model=model_dd.value,
                prompt_ver=prompt_ver_dd.value,
                code=code_in.value or "",
                tb=tb_in.value or "",
                question=question_in.value or "",
                file_path=active_file_path,
                num_predict=int(num_predict_sl.value),
                temperature=float(temp_sl.value),
                learning_card=bool(mode_dd.value == "Debug" and learning_card_ck.value),
                output_callback=lambda delta: print(delta, end="", flush=True),
            )

        fixed = result.get("fixed_code", "").strip()
        apply_btn.disabled = (fixed == "")
        apply_file_btn.disabled = (fixed == "" or not active_file_path)

        if active_file_path:
            LAST_OUTPUT["loaded_file_path"] = active_file_path
            restore_file_btn.disabled = not Path(active_file_path + ".bak").exists()
            update_file_info(active_file_path, message="当前结果基于已加载文件生成。", ok=True)

        if result.get("cache_hit"):
            status.value = "<b style='color:green'>完成（缓存命中）✅</b>"
        elif fixed:
            if active_file_path:
                status.value = "<b style='color:green'>完成 ✅</b>（已检测到修复后代码，可点击 Apply 或 Apply to File）"
            else:
                status.value = "<b style='color:green'>完成 ✅</b>（已检测到修复后代码，可点击 Apply）"
        else:
            status.value = "<b style='color:green'>完成 ✅（未检测到修复后代码）</b>"

    except Exception as e:
        status.value = f"<b style='color:#b00'>失败：</b> {e}"
    finally:
        run_btn.disabled = False


In [11]:
# 绑定按钮事件
run_btn.on_click(on_run)
load_file_btn.on_click(on_load_file)
apply_btn.on_click(on_apply)
apply_file_btn.on_click(on_apply_file)
restore_file_btn.on_click(on_restore_backup)
undo_btn.on_click(on_undo)
clear_btn.on_click(on_clear)
export_btn.on_click(on_export)
report_btn.on_click(on_export_report)
compare_btn.on_click(on_compare_prompt)
demo_btn.on_click(on_fill_demo)
mode_dd.observe(on_mode_change, names='value')


In [12]:
# ===================== 最终 UI =====================
ui = W.VBox(
    [
        header_box,
        control_panel,
        workspace
    ],
    layout=W.Layout(width='100%')
)

with out:
    print("Welcome to Coldrain's ColdCode")
    print("在左侧输入问题、代码和报错信息，然后点击 Run。")
    print("新增：ROCm Doctor、错误成长卡、Prompt Compare、Fill Demo。")

with prompt_lab_out:
    print("点击 Prompt Compare 查看当前模式在 v0-v3 下的提示词差异。")

on_mode_change()

display(ui)


请帮我修复，给最小步骤和 diff。

def add(a, b):
    return a + b

print(ad(1, 2))

Traceback (most recent call last):
  File "demo.py", line 4, in <module>
    print(ad(1, 2))
NameError: name 'ad' is not defined

---

你也可以试试新的 ROCm Doctor 模式，示例问题：

rocminfo 能看到 GPU，但 `python -c "import torch; print(torch.version.hip); print(torch.cuda.is_available())"` 输出 `None` 和 `False`，这更像是哪里出了问题？
